# **🎬 Movie Recommendation System Part 2**

## **I. Introduction**

### **i. Context**

Birds of a feather flock together—and so do people with similar tastes in movies.

Word-of-mouth has long been one of the most trusted sources of movie recommendations. Phrases like “my friends said it was good” often carry more weight than professional reviews, largely because shared preferences built over time create mutual trust. This intuition is reflected in my own experience. After spending three hours watching and discussing Legends of the Fall with a close friend, he was surprised by how closely our movie preferences aligned. Since then, we have relied heavily on each other’s recommendations, which have consistently matched our tastes.

While such human-based recommendations can be remarkably accurate, they do not scale. Online streaming platforms face a fundamentally different challenge: users simply do not have enough friends to explore the vast and ever-growing catalogs available today. Platforms like Netflix operate at a massive scale, serving millions of users and hosting tens of thousands of titles. In this context, automated recommendation systems become essential.

By leveraging users’ historical interactions with movies, a recommendation system can identify patterns in preferences and suggest relevant content. This not only enhances user satisfaction but also increases user engagement and, ultimately, platform revenue.

Although this project focuses on movie recommendations, the techniques explored here are broadly applicable. The same principles can be extended to recommend any type of item—such as music, products, or articles—where user preference data is available.

### **ii. Objective**

In this case study, we build and compare several recommendation system approaches:

- **Clustering-based recommendation system**
- **Content-based collaborative filtering**

To demonstrate these techniques, we use a **movie ratings dataset**, which captures users’ historical interactions with items and serves as the foundation for modeling user preferences.

### **iii. Dataset**

The **ratings** dataset consists of the following attributes:

- **userId**: A unique identifier for each user  
- **movieId**: A unique identifier for each movie  
- **rating**: The rating assigned by a user to a movie  
- **timestamp**: The time at which the rating was recorded

- **movies** dataset - This dataset contains the following attributes:
    - movieId
    - title
    - genres

- **tags** dataset- This dataset contains the following attributes:
    - userId
    - movieId
    - tag
    - timestamp

### **iv. Libraries**

In [1]:
# I. ----- Introduction -----
# This chapter does not have Python code.

# II. ----- Data Overview -----
import pandas as pd
# Surprise
from surprise import accuracy
from surprise.reader import Reader
from surprise.dataset import Dataset
from surprise.model_selection import GridSearchCV
from surprise.model_selection import train_test_split
from surprise import CoClustering

# III. ----- Modelling -----
from collections import defaultdict
import numpy as np
import nltk
import re
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## **II. Data Overview**

### **i. Data Retrieval**

In [2]:
# Import dataset
movies = pd.read_csv('movies.csv')
# First five row
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
# Shape of the DataFrame
movies.shape

(9742, 3)

In [4]:
# Import dataset
ratings = pd.read_csv('ratings.csv')

In [5]:
# Shape of the ratings dataset
ratings.shape

(100836, 4)

### **ii. Data Preprocessing**

In [6]:
# Merge on movieID
ratings_with_title = pd.merge(ratings, movies[['movieId', 'title']], on='movieId', how = 'inner')
# First five rows
ratings_with_title.head()

,userId,movieId,rating,timestamp,title
0,1,1,4.0,964982703,Toy Story (1995)
1,5,1,4.0,847434962,Toy Story (1995)
2,7,1,4.5,1106635946,Toy Story (1995)
3,15,1,2.5,1510577970,Toy Story (1995)
4,17,1,4.5,1305696483,Toy Story (1995)


In [7]:
ratings_with_title.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 100836 entries, 0 to 100835
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
 4   title      100836 non-null  object 
dtypes: float64(1), int64(3), object(1)
memory usage: 4.6+ MB


🔬 **Observations**

- There are **100,836 observations** and **5 columns** in the data.
- All the columns are of **numeric data type** except the **title column**. The title column is of **object data type**.
- The data type of the timestamp column is int64 which is not correct. We can convert this to DateTime format but **we don't need a timestamp for our analysis**. Hence, **we can drop this column**.

In [8]:
# Dropping the timestamp column
rating = ratings_with_title.drop(['timestamp'], axis=1)

In [9]:
# Average ratings
average_rating = rating.groupby('movieId')['rating'].mean()
# Count of ratings
count_rating = rating.groupby('movieId')['rating'].count()
# Count and average of ratings
final_rating = pd.DataFrame({'avg_rating': average_rating, 'rating_count': count_rating})

In [10]:
# First firve rows
final_rating.head()

,avg_rating,rating_count
movieId,,
1,3.920930,215
2,3.431818,110
3,3.259615,52
4,2.357143,7
5,3.071429,49


### **ii. Data Validation Check**

#### **1. Number of Unique Users**

In [11]:
ratings_with_title['userId'].nunique()

610

#### **2. Number of Unique Movies**

In [12]:
# Find the number of unique movies
ratings_with_title['title'].nunique()

9719

## **III. Modelling**

### **i. Preparation**

#### **1. Function Preparation**

In [13]:
def precision_recall_at_k(model, k = 10, threshold = 3.5):

    # Map prediction to user
    user_est_true = defaultdict(list)
    # Make predictions on the test data
    predictions=model.test(testset)
    # Get true users
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))
    # Initialize precision and recall dictionary
    precisions = dict()
    recalls = dict()
    for uid, user_ratings in user_est_true.items():
        # Sort user ratings by estimated value
        user_ratings.sort(key = lambda x: x[0], reverse = True)
        # Number of relevant items
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        # Number of recommended items in top k
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[ : k])
        # Number of relevant and recommended items in top k
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold))
                              for (est, true_r) in user_ratings[ : k])
        # Precision@K: Proportion of recommended items that are relevant
        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0 # When n_rec_k is 0, Precision is undefined. We here set Precision to 0 when n_rec_k is 0
        # Recall@K: Proportion of relevant items that are recommended
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0 # When n_rel is 0, Recall is undefined. We here set Recall to 0 when n_rel is 0
    # Mean of all the predicted precisions are calculated
    precision = round((sum(prec for prec in precisions.values()) / len(precisions)), 3)
    # Mean of all the predicted recalls are calculated
    recall = round((sum(rec for rec in recalls.values()) / len(recalls)), 3)
    # RMSE
    accuracy.rmse(predictions)

    print('Precision: ', precision)
    print('Recall: ', recall)
    print('F_1 score: ', round((2 * precision * recall) / (precision + recall), 3))

#### **2. Dataset Conversion**

Below, we are loading the **`rating` dataset**, which is a **pandas DataFrame**, into a **different format called `surprise.dataset.DatasetAutoFolds`**. This is required by this library. To do this we will be **using the classes `Reader` and `Dataset`**.

In [14]:
# Instantiate Reader scale with expected rating scale
reader = Reader(rating_scale = (0, 5))
# Load the rating dataset
data = Dataset.load_from_df(rating[['userId', 'movieId', 'rating']], reader)
# Split the data into train and test dataset
trainset, testset = train_test_split(data, test_size = 0.2, random_state = 42)

### **ii. Model 1: Cluster-Based Recommendation System**

#### **1. Initial Modelling**

In [15]:
clust_baseline = CoClustering(random_state = 1)
# Train
clust_baseline.fit(trainset)
# Compute precision@k, recall@k, and F_1 score with k = 10
precision_recall_at_k(clust_baseline)

RMSE: 0.9490
Precision:  0.717
Recall:  0.502
F_1 score:  0.591


🔬 **Observations**

- **RMSE:** 0.9490  
  Indicates moderate prediction error when estimating user ratings; the model captures general user preferences but has room for improvement.

- **Precision:** 0.717  
  About 72% of the recommended movies were relevant, showing the system is fairly accurate in suggesting items users like.

- **Recall:** 0.502  
  The model retrieves roughly 50% of all relevant movies, suggesting it misses some items a user might enjoy.

- **F1 Score:** 0.591  
  Balances precision and recall; the moderate F1 indicates a trade-off between recommending relevant movies and covering all relevant items.

**Interpretation:**  
The clustering-based approach provides reasonable accuracy in recommending popular or similar movies within clusters. While precision is strong, recall is lower, indicating the system may fail to surface all relevant items for users. This suggests potential improvements could come from hybrid approaches or finer-grained clustering.

**Case Study: `userId = 4` `movieId = 10`**

In [16]:
clust_baseline.predict(4, 10, r_ui = 4, verbose = True)

user: 4          item: 10         r_ui = 4.00   est = 3.68   {'was_impossible': False}


Prediction(uid=4, iid=10, r_ui=4, est=3.6757402992691386, details={'was_impossible': False})

🔬 **Observations**

- **User:** 4  
- **Item (Movie ID):** 10  
- **Actual Rating (r_ui):** 4.00  
- **Predicted Rating (est):** 3.68  
- **Prediction Validity:** Successful (`was_impossible=False`)

**Interpretation:**  
The model slightly underestimates the user’s actual rating by 0.32 points. This indicates that the recommendation system is reasonably accurate in predicting user preferences for individual items, though minor deviations exist. Continuous evaluation across more samples would provide a better assessment of overall model performance.

**Case Study: `userId = 4` `movieId = 3`**

In [17]:
clust_baseline.predict(4, 3, verbose = True)

user: 4          item: 3          r_ui = None   est = 3.26   {'was_impossible': False}


Prediction(uid=4, iid=3, r_ui=None, est=3.258169827544438, details={'was_impossible': False})

🔬 **Observations**

- **User:** 4  
- **Item (Movie ID):** 3  
- **Actual Rating (r_ui):** None (user has not rated this item)  
- **Predicted Rating (est):** 3.26  
- **Prediction Validity:** Successful (`was_impossible=False`)

**Interpretation:**  
The model predicts a rating of 3.26 for an item the user has not yet rated. This demonstrates the recommendation system’s ability to generate **personalized predictions for unseen items**, enabling the system to suggest movies that the user may enjoy even without prior interaction. Such predictions are fundamental for recommending new or unrated items to users.

#### **2. Hyperparameter Tuning**

**Hyperparameter Overview**

- **n_cltr_u** (int) – Number of **user clusters**. The default value is 3.
- **n_cltr_i** (int) – Number of **item clusters**. The default value is 3.
- **n_epochs** (int) – Number of **iteration of the optimization loop**. The default value is 3.
- **random_state** (int, RandomState instance from NumPy, or None) – Determines the RNG that will be used for initialization. If int, random_state will be used as a seed for a new RNG. This is useful to get the same initialization over multiple calls to fit(). If RandomState instance, this same instance is used as RNG. If None, the current RNG from NumPy is used. The default value is None.
- **verbose** (bool) – If True, the current epoch will be printed. The default value is False.

In [18]:
# Set the parameter space to tune
param_grid = {'n_cltr_u': [3, 4, 5, 6], 'n_cltr_i': [3, 4, 5, 6], 'n_epochs': [30, 40, 50]}
# 3-Fold gridsearch cross-validation
gs = GridSearchCV(CoClustering, param_grid, measures = ['rmse'], cv = 3, n_jobs = -1)
# Fit data
gs.fit(data)
# Print the best RMSE score
print(gs.best_score['rmse'])
# Print the combination of parameters that gives the best RMSE score
print(gs.best_params['rmse'])

0.9543068744686257
{'n_cltr_u': 5, 'n_cltr_i': 3, 'n_epochs': 30}


#### **3. Final Modelling**

In [19]:
clust_tuned = CoClustering(n_cltr_u = 3, n_cltr_i = 3, n_epochs = 30, random_state = 1)
# Train the algorithm on the train set
clust_tuned.fit(trainset)
# Compute precision@k, recall@k, and F_1 score with k = 10
precision_recall_at_k(clust_tuned)

RMSE: 0.9499
Precision:  0.715
Recall:  0.5
F_1 score:  0.588


🔬 **Observations**

- **RMSE:** 0.9499  
  Slight increase in prediction error compared to the baseline (0.9490), indicating minimal impact on overall rating accuracy.  

- **Precision:** 0.715  
  Very similar to baseline (0.717), showing the tuned model maintains strong relevance in recommended items.  

- **Recall:** 0.500  
  Slight decrease from baseline (0.502), suggesting the model retrieves roughly half of the relevant items for each user.  

- **F1 Score:** 0.588  
  Marginally lower than baseline (0.591), reflecting the trade-off between precision and recall after tuning.

**Interpretation:**  
Hyperparameter tuning led to **minimal changes in performance**, preserving precision while slightly reducing recall and F1. This suggests the baseline clustering configuration was already close to optimal, and further improvements may require alternative modeling approaches or hybrid recommendation techniques.

**Case Study: `userId = 4` `movieId = 10`**

In [20]:
clust_tuned.predict(4, 10, r_ui = 4, verbose = True)

user: 4          item: 10         r_ui = 4.00   est = 3.65   {'was_impossible': False}


Prediction(uid=4, iid=10, r_ui=4, est=3.6549811878747214, details={'was_impossible': False})

🔬 **Observations**

- **User:** 4  
- **Item (Movie ID):** 10  
- **Actual Rating (r_ui):** 4.00  
- **Predicted Rating (est):** 3.65  
- **Prediction Validity:** Successful (`was_impossible=False`)

**Interpretation:**  
The tuned model slightly underestimates the user’s rating by 0.35 points. This shows the recommendation system remains reasonably accurate for individual predictions, even after hyperparameter optimization. Such predictions highlight the model’s capability to provide personalized recommendations while maintaining stability in estimated ratings.

**Case Study: `userId = 4` `movieId = 3`**

In [21]:
# Using Co-clustering based optimized model
clust_tuned.predict(4, 3, verbose = True)

user: 4          item: 3          r_ui = None   est = 3.23   {'was_impossible': False}


Prediction(uid=4, iid=3, r_ui=None, est=3.2318175022354345, details={'was_impossible': False})

🔬 **Observations**

- **User:** 4  
- **Item (Movie ID):** 3  
- **Actual Rating (r_ui):** None (user has not rated this item)  
- **Predicted Rating (est):** 3.23  
- **Prediction Validity:** Successful (`was_impossible=False`)

**Interpretation:**  
The model predicts a rating of 3.23 for an item the user has not previously rated. This demonstrates the system’s ability to generate **personalized recommendations for unseen items**, supporting discovery of new movies for the user. The slight decrease in predicted value compared to pre-tuning (3.26 → 3.23) indicates hyperparameter tuning subtly adjusted the model’s confidence without compromising relevance.

#### **4. Implementation**

Below we will be implementing a function where the input parameters are:

- data: A **rating** dataset
- user_id: A user id **against which we want the recommendations**
- top_n: The **number of movies we want to recommend**
- algo: The algorithm we want to use **for predicting the ratings**
- The output of the function is a **set of top_n items** recommended for the given user id based on the given algorithm

In [22]:
def get_recommendations(data, user_id, top_n, algo):

    # Initialize recommendations
    recommendations = []
    # Create an user-item interactions matrix
    user_item_interactions_matrix = data.pivot(index = 'userId', columns = 'movieId', values = 'rating')
    # Extract those movie IDs which the userId has not interacted yet
    non_interacted_movies = user_item_interactions_matrix.loc[user_id][user_item_interactions_matrix.loc[user_id].isnull()].index.tolist()
    # Loop through each of the movie IDs which userId has not interacted yet
    for item_id in non_interacted_movies:
        # Predict the ratings for those non interacted movie IDs by this user
        est = algo.predict(user_id, item_id).est
        # Append the predicted ratings
        recommendations.append((item_id, est))
    # Sort the predicted ratings in descending order
    recommendations.sort(key = lambda x: x[1], reverse = True)

    return recommendations[:top_n]

In [23]:
# Top 5 recommendations for userId 4 using Co-clustering based optimized algorithm
clustering_recommendations = get_recommendations(rating, 4, 5, clust_tuned)

#### **5. Rating Correction and Movie Ranking**

When comparing movies based solely on user ratings, the **number of users who rated a movie** is equally important. A high rating with very few ratings may not reflect broad user preference, while a slightly lower rating with many ratings indicates wider approval.  

To address this, we compute **corrected ratings** for each movie. Empirically, the likelihood of a movie being well-liked is considered **directly proportional to the inverse of the square root of its rating count**.  

**Example:**  
- Movie A: rating = 4, rating_count = 3 → less broadly liked  
- Movie B: rating = 3, rating_count = 50 → more widely liked  

This approach ensures that movies with higher engagement are appropriately weighted in recommendation decisions, balancing **rating score** with **popularity** for more robust predictions.

In [24]:
def ranking_movies(recommendations, final_rating):

    # Sort the movies based on ratings count
    ranked_movies = final_rating.loc[[items[0] for items in recommendations]].sort_values('rating_count', ascending = False)[['rating_count']].reset_index()
    # Merge with the recommended movies to get predicted ratings
    ranked_movies = ranked_movies.merge(pd.DataFrame(recommendations, columns = ['movieId', 'predicted_ratings']), on = 'movieId', how = 'inner')
    # Rank the movies based on corrected ratings
    ranked_movies['corrected_ratings'] = ranked_movies['predicted_ratings'] - 1 / np.sqrt(ranked_movies['rating_count'])
    # Sort the movies based on corrected ratings
    ranked_movies = ranked_movies.sort_values('corrected_ratings', ascending = False)

    return ranked_movies

**Note:** In the **above-corrected rating formula**, we can add the **quantity `1 / np.sqrt(n)` instead of subtracting it to get more optimistic predictions**. But here we are **subtracting this quantity**, as there are some movies with ratings of 5 and **we can't have a rating more than 5 for a movie**.

In [25]:
# Ranking movies based on the above recommendations
ranking_movies(clustering_recommendations, final_rating)

,movieId,rating_count,predicted_ratings,corrected_ratings
0,304,3,5,4.422650
1,53,2,5,4.292893
2,99,2,5,4.292893
3,238,2,5,4.292893
4,148,1,5,4.000000


### **iii. Model 2: Content-Based Recommendation System**

#### **1. Data Preparation**

In [26]:
# Import the tags data
tags = pd.read_csv('tags.csv')
# Top 5 rows
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


The dataset does not include detailed movie reviews or plot summaries. To construct meaningful content-based features, we combine the available textual attributes from multiple sources:

- `title`
- `genres`
- `tags`

These fields are concatenated to create a unified text representation for each movie.  

We then apply **TF-IDF (Term Frequency–Inverse Document Frequency)** to transform the combined text into numerical feature vectors. This approach captures the relative importance of words across the movie corpus while reducing the influence of commonly occurring terms.

The resulting TF-IDF vectors are used to compute similarity scores between movies, enabling the system to recommend items with similar content characteristics.

In [27]:
# Merge all the three datasets on movieId
ratings_with_title = pd.merge(ratings, movies[['movieId', 'title', 'genres']], on = 'movieId' )
final_ratings = pd.merge(ratings_with_title, tags[['movieId', 'tag']], on = 'movieId' )

final_ratings

,userId,movieId,rating,timestamp,title,genres,tag
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
1,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
2,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,fun
3,5,1,4.0,847434962,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
4,5,1,4.0,847434962,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
...,...,...,...,...,...,...,...
233208,599,176419,3.5,1516604655,Mother! (2017),Drama|Horror|Mystery|Thriller,uncomfortable
233209,599,176419,3.5,1516604655,Mother! (2017),Drama|Horror|Mystery|Thriller,unsettling
233210,594,7023,4.5,1108972356,"Wedding Banquet, The (Xi yan) (1993)",Comedy|Drama|Romance,In Netflix queue
233211,606,6107,4.0,1171324428,Night of the Shooting Stars (Notte di San Lore...,Drama|War,World War II


🔬 **Observations**

After merging the ratings, movies, and tags datasets, the resulting DataFrame includes the following columns:

- `userId`
- `movieId`
- `rating`
- `timestamp`
- `title`
- `genres`
- `tag`

1. **Row Expansion Due to Tags**  
   The dataset now contains significantly more rows (233,213 entries) compared to the original ratings dataset (~100K rows).  
   This increase occurs because movies with multiple tags generate multiple rows per `(userId, movieId)` pair.

2. **Duplicate Rating Entries**  
   Identical ratings appear multiple times when a movie has multiple associated tags.  
   For example, *Toy Story (1995)* appears repeatedly with different tag values (e.g., "pixar", "fun"), even though the rating itself is unchanged.

3. **One-to-Many Relationship**  
   The merge introduces a **one-to-many relationship** between ratings and tags:
   - One rating → multiple tag entries  
   - Enables richer content-based feature construction  

4. **Enhanced Textual Information**  
   The addition of `genres` and `tag` provides valuable metadata for:
   - Content-based recommendation systems  
   - TF-IDF feature extraction  
   - Similarity computation between movies  

5. **Potential Need for Aggregation**  
   Since ratings are duplicated across tags, care must be taken during modeling:
   - For collaborative filtering → deduplicate `(userId, movieId)` pairs  
   - For content-based modeling → aggregate or concatenate tags per movie  

**Interpretation**

The merged dataset significantly enriches movie metadata by incorporating tags, enabling more advanced content-based recommendation techniques. However, the expanded row count and duplicate rating entries require careful preprocessing to avoid bias in model training and evaluation.

In [28]:
# Replace | character with space in genres column
final_ratings['genres'] = final_ratings['genres'].apply(lambda x: " ".join(x.split('|')))

In [29]:
# Combine title, genres, and tag columns
final_ratings['text'] = final_ratings['title'] + ' ' + final_ratings['genres'] + ' ' + final_ratings['tag']

final_ratings.head()

,userId,movieId,rating,timestamp,title,genres,tag,text
0,1,1,4.0,964982703,Toy Story (1995),Adventure Animation Children Comedy Fantasy,pixar,Toy Story (1995) Adventure Animation Children ...
1,1,1,4.0,964982703,Toy Story (1995),Adventure Animation Children Comedy Fantasy,pixar,Toy Story (1995) Adventure Animation Children ...
2,1,1,4.0,964982703,Toy Story (1995),Adventure Animation Children Comedy Fantasy,fun,Toy Story (1995) Adventure Animation Children ...
3,5,1,4.0,847434962,Toy Story (1995),Adventure Animation Children Comedy Fantasy,pixar,Toy Story (1995) Adventure Animation Children ...
4,5,1,4.0,847434962,Toy Story (1995),Adventure Animation Children Comedy Fantasy,pixar,Toy Story (1995) Adventure Animation Children ...


Now, we will **keep only four columns** - userId, movieId, rating, and text. We will drop the duplicate titles from the data and make it the **title column as the index** of the dataframe.

In [30]:
# Create the final_ratings dataset with specified columns
final_ratings = final_ratings[['userId', 'movieId', 'rating', 'title', 'text']]
# Drop the duplicate records
final_ratings = final_ratings.drop_duplicates(subset = ['title'])
# Set the index
final_ratings = final_ratings.set_index('title')
# See the first five records of the dataset
final_ratings.head()

,userId,movieId,rating,text
title,,,,
Toy Story (1995),1,1,4.0,Toy Story (1995) Adventure Animation Children ...
Grumpier Old Men (1995),1,3,4.0,Grumpier Old Men (1995) Comedy Romance moldy
Seven (a.k.a. Se7en) (1995),1,47,5.0,Seven (a.k.a. Se7en) (1995) Mystery Thriller m...
"Usual Suspects, The (1995)",1,50,5.0,"Usual Suspects, The (1995) Crime Mystery Thril..."
Bottle Rocket (1996),1,101,5.0,Bottle Rocket (1996) Adventure Comedy Crime Ro...


In [31]:
final_ratings.shape

(1554, 4)

#### **2. Text Data Handling Preparation**

In [32]:
nltk.download('omw-1.4')
# Download punctuations
nltk.download('punkt')
nltk.download('punkt_tab')
# Download stopwords
nltk.download('stopwords')
# Download wordnet
nltk.download('wordnet')

[nltk_data] Downloading package omw-1.4 to /Users/cj/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /Users/cj/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/cj/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/cj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/cj/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [33]:
def tokenize(text):

    # Make each letter as lowercase and removing non-alphabetical text
    text = re.sub(r"[^a-zA-Z]"," ", text.lower())
    # Extract each word in the text
    tokens = word_tokenize(text)
    # Remove stopwords
    words = [word for word in tokens if word not in stopwords.words("english")]
    # Lemmatize the words
    text_lems = [WordNetLemmatizer().lemmatize(lem).strip() for lem in words]

    return text_lems

#### **3. Feature Extraction (TF-IDF)**

In [34]:
# Create the TF-IDF object
tfidf = TfidfVectorizer(tokenizer = tokenize)

movie_tfidf = tfidf.fit_transform(final_ratings['text'].values).toarray()

/opt/anaconda3/envs/surprise/lib/python3.9/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [35]:
# Make the DataFrame of movie_tfidf data
pd.DataFrame(movie_tfidf)

,0,1,2,3,4,5,6,7,8,9,...,2774,2775,2776,2777,2778,2779,2780,2781,2782,2783
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1549,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1550,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1551,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1552,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [36]:
# Calculate the cosine similarity
similar_movies = cosine_similarity(movie_tfidf, movie_tfidf)

similar_movies

array([[1.        , 0.02268393, 0.        , ..., 0.02022472, 0.        ,
        0.        ],
       [0.02268393, 1.        , 0.        , ..., 0.04779055, 0.        ,
        0.        ],
       [0.        , 0.        , 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.02022472, 0.04779055, 0.        , ..., 1.        , 0.00719396,
        0.19617374],
       [0.        , 0.        , 0.        , ..., 0.00719396, 1.        ,
        0.01217017],
       [0.        , 0.        , 0.        , ..., 0.19617374, 0.01217017,
        1.        ]])

#### **4. Function Preparation**

In [37]:
# Function that takes in movie title as input and returns the top 10 recommended movies
def recommendations(title, similar_movies):

    recommended_movies = []
    indices = pd.Series(final_ratings.index)
    # Get the index of the movie that matches the title
    idx = indices[indices == title].index[0]
    # Create a Series with the similarity scores in descending order
    score_series = pd.Series(similar_movies[idx]).sort_values(ascending = False)
    # Get the indices of 10 most similar movies
    top_10_indexes = list(score_series.iloc[1 : 11].index)
    
    print(top_10_indexes)

    # Populate the list with the titles of the best 10 matching movies
    for i in top_10_indexes:
        recommended_movies.append(list(final_ratings.index)[i])

    return recommended_movies

In [38]:
recommendations('Usual Suspects, The (1995)', similar_movies)

[71, 1186, 124, 551, 569, 77, 719, 766, 123, 658]


['Game, The (1997)',
 'Andalusian Dog, An (Chien andalou, Un) (1929)',
 'Town, The (2010)',
 'Now You See Me (2013)',
 'Charade (1963)',
 'Negotiator, The (1998)',
 'Following (1998)',
 '21 Grams (2003)',
 'Inception (2010)',
 'Insomnia (2002)']

🔬 **Observations**

**Input Movie:** *Usual Suspects, The (1995)*  

**Top 10 Recommended Movies:**

1. Game, The (1997)  
2. Andalusian Dog, An (Chien andalou, Un) (1929)  
3. Town, The (2010)  
4. Now You See Me (2013)  
5. Charade (1963)  
6. Negotiator, The (1998)  
7. Following (1998)  
8. 21 Grams (2003)  
9. Inception (2010)  
10. Insomnia (2002)  

- **Strong Genre & Theme Alignment:**  
  Many recommended movies fall within the **crime, mystery, psychological thriller, and suspense** categories — consistent with the themes of *The Usual Suspects*.

- **Psychological & Twisting Narratives:**  
  Films such as *Inception*, *Insomnia*, *Following*, and *The Game* share complex storytelling, unreliable perspectives, or suspense-driven plots.

- **Heist / Crime Overlap:**  
  Movies like *The Town*, *Now You See Me*, and *The Negotiator* align closely with crime and deception-based themes.

- **Temporal Diversity:**  
  The recommendations span multiple decades (1929–2013), indicating the model prioritizes **content similarity over recency bias**.

- **Semantic Matching via TF-IDF:**  
  Since this model relies on textual features (`genres` + `tags`), similarity is driven by shared keywords and thematic descriptors rather than user behavior patterns.

**Interpretation**

The content-based recommendation system successfully identifies movies with similar thematic and genre characteristics. The recommendations demonstrate strong semantic alignment with the input movie, validating that TF-IDF feature extraction effectively captures meaningful textual relationships between films.

## **IV. Conclusion**

In this case study, we developed and evaluated two distinct recommendation system approaches: a **clustering-based recommendation model** and a **content-based recommendation system**. Both methods were implemented using a movie ratings dataset enriched with metadata such as genres and user-generated tags.

The clustering-based model leveraged user interaction patterns to group similar users or items and generate recommendations based on shared behavior. It achieved solid predictive performance (RMSE ≈ 0.95) with relatively strong precision, indicating that recommended items were often relevant. However, recall remained moderate, suggesting that some relevant items were not consistently surfaced.

The content-based recommendation system relied on textual features constructed from movie titles, genres, and tags. Using TF-IDF vectorization and similarity computation, the model successfully identified semantically similar movies. The recommendations demonstrated strong thematic alignment with input movies, validating the effectiveness of text-based feature engineering.

**Key Insights**

- **Collaborative (Clustering) Approach**
  - Strength: Captures collective user behavior patterns.
  - Limitation: Performance depends heavily on interaction density.
  - Better suited when rich user–item interaction data is available.

- **Content-Based Approach**
  - Strength: Does not rely on other users’ behavior.
  - Handles user cold-start scenarios effectively.
  - Limitation: May produce less diverse recommendations.

**Overall Takeaways**

This case study highlights the importance of:
- Proper data merging and preprocessing.
- Feature engineering for both interaction-based and text-based models.
- Evaluating recommendation systems using both regression metrics (RMSE) and classification-style metrics (Precision, Recall, F1).

While both models produced meaningful recommendations, further improvements could include:
- Hybrid recommendation systems combining collaborative and content-based signals.
- Dimensionality reduction techniques for improved scalability.
- Incorporation of additional metadata such as plot summaries or review text.
- Advanced modeling approaches such as matrix factorization or neural collaborative filtering.

Overall, this project demonstrates practical implementation, evaluation, and comparison of recommendation system methodologies, providing a strong foundation for scalable, production-ready recommender solutions.